# Inflation Signal Monitor — Colab Edition

This notebook downloads current public data from FRED each time it runs. It provides:

- the five-year nominal–real–breakeven decomposition;
- WTI oil, VIX, high-yield spreads, and the broad dollar;
- normalized cross-market comparisons;
- daily-change correlations;
- automatic identification of the largest breakeven movements; and
- CSV and HTML exports.

The breakeven rate measures inflation compensation. It is not a pure expectation because it can include inflation-risk and TIPS-liquidity premiums.

In [1]:
%pip -q install pandas plotly jinja2

## 1. Configuration

Change `ANALYSIS_START` if you want a different sample. The download begins in 2020 so you can move the analysis start backward without changing the data code.

In [2]:
DOWNLOAD_START = "2020-01-01"
ANALYSIS_START = "2025-09-01"   # Change to "2020-01-01" for the full sample
TOP_EPISODES = 12

SERIES = {
    "T5YIE": "5Y breakeven (%)",
    "DGS5": "5Y nominal yield (%)",
    "DFII5": "5Y real yield (%)",
    "DCOILWTICO": "WTI crude ($/barrel)",
    "VIXCLS": "VIX",
    "BAMLH0A0HYM2": "US high-yield spread (%)",
    "DTWEXBGS": "Broad dollar index",
}
COLORS = {
    "nominal": "#D9E2F1", "real": "#3B82F6", "breakeven": "#F2B84B",
    "oil": "#E8793E", "vix": "#B565D9", "hy": "#EF6461", "dollar": "#43AA8B",
}

## 2. Download and prepare the FRED data

No FRED API key is required. Values from slower-updating series are carried forward for at most five Treasury observation days for chart alignment.

In [3]:
from getpass import getpass
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

# The key is entered privately and is not displayed or saved in the notebook.
from google.colab import userdata

FRED_API_KEY = userdata.get("FRED_API_KEY")

if not FRED_API_KEY:
    raise ValueError(
        "The Colab secret FRED_API_KEY was not found or is empty."
    )


def fred(series_id, start=DOWNLOAD_START):
    """Download one series from the official FRED observations API."""

    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start,
        "sort_order": "asc",
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=(5, 20)       # 5-second connection, 20-second read limit
        )
        response.raise_for_status()

    except requests.Timeout as exc:
        raise RuntimeError(
            f"FRED timed out while downloading {series_id}. "
            "Wait briefly and rerun this cell."
        ) from exc

    except requests.RequestException as exc:
        raise RuntimeError(
            f"FRED request failed for {series_id}: {exc}"
        ) from exc

    payload = response.json()

    if "error_code" in payload:
        raise RuntimeError(
            f"FRED rejected the request for {series_id}: "
            f"{payload.get('error_message', payload)}"
        )

    observations = payload.get("observations", [])

    if not observations:
        raise RuntimeError(
            f"FRED returned no observations for {series_id}."
        )

    frame = pd.DataFrame(observations)

    frame["date"] = pd.to_datetime(
        frame["date"],
        errors="coerce"
    )

    frame[series_id] = pd.to_numeric(
        frame["value"].replace(".", np.nan),
        errors="coerce"
    )

    return (
        frame.dropna(subset=["date"])
             .set_index("date")[series_id]
             .sort_index()
    )


# Test the connection with T5YIE before requesting everything else.
print("Testing the FRED connection with T5YIE ...")
test_series = fred("T5YIE")
print(
    f"T5YIE downloaded successfully: "
    f"{test_series.notna().sum():,} observations"
)

# Download the remaining series only after the test succeeds.
downloaded = {"T5YIE": test_series}

for series_id in SERIES:
    if series_id == "T5YIE":
        continue

    print(f"Downloading {series_id} ...")
    downloaded[series_id] = fred(series_id)

raw = (
    pd.concat(downloaded, axis=1)
      .sort_index()
      .loc[DOWNLOAD_START:]
)

# Use the T5YIE observation calendar and align the other series.
data = raw.loc[raw["T5YIE"].notna()].ffill(limit=5)

for series_id in SERIES:
    data[f"d_{series_id}"] = data[series_id].diff()

data["nominal_move_bp"] = 100 * data["d_DGS5"]
data["real_move_bp"] = 100 * data["d_DFII5"]
data["breakeven_move_bp"] = 100 * data["d_T5YIE"]

analysis = data.loc[ANALYSIS_START:].copy()

print(
    f"\nDownloaded {len(data):,} aligned observations: "
    f"{data.index.min().date()} to {data.index.max().date()}"
)

print(
    f"Analysis sample: {analysis.index.min().date()} "
    f"to {analysis.index.max().date()} "
    f"({len(analysis):,} observations)"
)

Testing the FRED connection with T5YIE ...
T5YIE downloaded successfully: 1,676 observations

Downloaded 1,676 aligned observations: 2020-01-02 to 2026-09-14
Analysis sample: 2025-09-02 to 2026-09-14 (259 observations)


## 3. Latest readings and decomposition check

In [4]:
latest = analysis.iloc[-1]
previous = analysis.iloc[-2]
summary = pd.DataFrame({
    "Latest": [latest[s] for s in SERIES],
    "Previous": [previous[s] for s in SERIES],
    "Daily change": [latest[s] - previous[s] for s in SERIES],
}, index=[SERIES[s] for s in SERIES])
summary.index.name = f"Observation date: {analysis.index[-1].date()}"
display(summary.style.format({"Latest":"{:.3f}", "Previous":"{:.3f}", "Daily change":"{:+.3f}"}))

identity_error = latest["DGS5"] - latest["DFII5"] - latest["T5YIE"]
print(f"\nLevel identity check, DGS5 − DFII5 − T5YIE: {identity_error:+.4f} percentage points")

,Latest,Previous,Daily change
Observation date: 2026-09-14,,,
5Y breakeven (%),2.400,2.400,+0.000
5Y nominal yield (%),4.780,4.780,+0.000
5Y real yield (%),2.380,2.380,+0.000
WTI crude ($/barrel),97.260,97.260,+0.000
VIX,17.100,15.840,+1.260
US high-yield spread (%),2.710,2.650,+0.060
Broad dollar index,118.213,118.213,+0.000



Level identity check, DGS5 − DFII5 − T5YIE: +0.0000 percentage points


## 4. Interactive five-year Treasury decomposition

Use the buttons above the graph or drag across the chart. Hovering reports exact observations.

In [5]:
fig_decomp = go.Figure()
for key, label, color in [
    ("DGS5", "Nominal yield", COLORS["nominal"]),
    ("DFII5", "Real yield", COLORS["real"]),
    ("T5YIE", "Breakeven", COLORS["breakeven"]),
]:
    fig_decomp.add_trace(go.Scatter(x=analysis.index, y=analysis[key], mode="lines", name=label,
                                    line=dict(color=color, width=2.2 if key=="T5YIE" else 1.7)))
fig_decomp.update_layout(
    title="Five-year Treasury decomposition: nominal = real + inflation compensation",
    template="plotly_dark", height=560, hovermode="x unified",
    yaxis_title="Percent", legend=dict(orientation="h", y=1.04),
    margin=dict(l=55,r=25,t=85,b=45),
)
fig_decomp.update_xaxes(rangeselector=dict(buttons=[
    dict(count=6,label="6M",step="month",stepmode="backward"),
    dict(count=1,label="1Y",step="year",stepmode="backward"),
    dict(count=3,label="3Y",step="year",stepmode="backward"),
    dict(step="all",label="Full"),
]), rangeslider=dict(visible=False))
fig_decomp.show()

## 5. Cross-market stress map

Each series is indexed to 100 at the beginning of the analysis sample. This compares paths, not units.

In [6]:
cross = analysis[["DCOILWTICO","VIXCLS","BAMLH0A0HYM2","DTWEXBGS"]].copy()
normalized = 100 * cross / cross.apply(lambda x: x.dropna().iloc[0])
fig_cross = go.Figure()
for key,label,color in [
    ("DCOILWTICO","WTI crude",COLORS["oil"]),
    ("VIXCLS","VIX",COLORS["vix"]),
    ("BAMLH0A0HYM2","High-yield spread",COLORS["hy"]),
    ("DTWEXBGS","Broad dollar",COLORS["dollar"]),
]:
    fig_cross.add_trace(go.Scatter(x=normalized.index,y=normalized[key],mode="lines",name=label,line=dict(color=color,width=1.8)))
fig_cross.update_layout(title="Cross-market signals, indexed to 100",template="plotly_dark",height=540,
                        yaxis_title="Index",hovermode="x unified",legend=dict(orientation="h",y=1.04),
                        margin=dict(l=55,r=25,t=85,b=45))
fig_cross.update_xaxes(rangeselector=dict(buttons=[
    dict(count=6,label="6M",step="month",stepmode="backward"),
    dict(count=1,label="1Y",step="year",stepmode="backward"),
    dict(count=3,label="3Y",step="year",stepmode="backward"),
    dict(step="all",label="Full"),
]))
fig_cross.show()

## 6. Correlations with daily breakeven changes

These are descriptive correlations, not structural effects.

In [7]:
change_columns = {
    "d_DCOILWTICO": "WTI crude",
    "d_VIXCLS": "VIX",
    "d_BAMLH0A0HYM2": "High-yield spread",
    "d_DTWEXBGS": "Broad dollar",
    "d_DGS5": "Nominal yield",
    "d_DFII5": "Real yield",
}
corr = analysis[["d_T5YIE", *change_columns]].corr().loc["d_T5YIE", list(change_columns)]
corr.index = [change_columns[x] for x in corr.index]
corr_table = corr.rename("Correlation with Δ5Y breakeven").to_frame()
corr_table = corr_table.sort_values("Correlation with Δ5Y breakeven", ascending=False)
display(corr_table.style.format("{:+.3f}").background_gradient(cmap="RdYlGn", vmin=-1, vmax=1))

sorted_corr = corr.sort_values()
fig_corr = px.bar(x=sorted_corr.values, y=sorted_corr.index, orientation="h",
                  labels={"x":"Correlation","y":""}, title="Daily-change correlations")
fig_corr.update_traces(marker_color=[COLORS["hy"] if x<0 else COLORS["dollar"] for x in sorted_corr.values])
fig_corr.update_layout(template="plotly_dark",height=420,showlegend=False,xaxis_range=[-1,1])
fig_corr.show()

,Correlation with Δ5Y breakeven
WTI crude,+0.600
Nominal yield,+0.573
Broad dollar,+0.182
VIX,+0.162
High-yield spread,+0.004
Real yield,-0.037


## 7. Largest breakeven-move episodes

The configuration labels summarize the signs of the real-yield and breakeven components. They are descriptive labels, not causal classifications.

In [8]:
def configuration(row):
    real, be = row["real_move_bp"], row["breakeven_move_bp"]
    if real > 0 and be > 0: return "Growth + inflation"
    if real > 0 and be < 0: return "Tighter real conditions"
    if real < 0 and be > 0: return "Supply / stagflation"
    return "Demand weakness"

episodes = analysis.dropna(subset=["breakeven_move_bp"]).copy()
episodes["absolute_move"] = episodes["breakeven_move_bp"].abs()
episodes["configuration"] = episodes.apply(configuration, axis=1)
episodes = episodes.nlargest(TOP_EPISODES,"absolute_move")[[
    "breakeven_move_bp","nominal_move_bp","real_move_bp","DCOILWTICO","VIXCLS","BAMLH0A0HYM2","DTWEXBGS","configuration"
]]
episodes.index.name = "date"
display(episodes.style.format({
    "breakeven_move_bp":"{:+.1f}","nominal_move_bp":"{:+.1f}","real_move_bp":"{:+.1f}",
    "DCOILWTICO":"${:.2f}","VIXCLS":"{:.2f}","BAMLH0A0HYM2":"{:.2f}%","DTWEXBGS":"{:.2f}"
}).background_gradient(subset=["breakeven_move_bp"],cmap="RdYlGn"))

,breakeven_move_bp,nominal_move_bp,real_move_bp,DCOILWTICO,VIXCLS,BAMLH0A0HYM2,DTWEXBGS,configuration
date,,,,,,,,
2025-10-23 00:00:00,+10.0,+5.0,-5.0,$62.44,17.30,2.96%,120.95,Supply / stagflation
2026-03-23 00:00:00,-10.0,-6.0,+4.0,$89.33,26.15,3.19%,119.94,Tighter real conditions
2026-05-06 00:00:00,-9.0,-9.0,+0.0,$98.75,17.39,2.75%,118.10,Demand weakness
2026-07-29 00:00:00,+8.0,+2.0,-6.0,$86.08,20.66,2.87%,120.79,Supply / stagflation
2026-05-20 00:00:00,-7.0,-10.0,-3.0,$101.69,17.44,2.80%,119.16,Demand weakness
2026-03-02 00:00:00,+6.0,+11.0,+5.0,$71.13,21.44,3.03%,118.67,Growth + inflation
2026-07-27 00:00:00,-6.0,-3.0,+3.0,$84.25,18.67,2.81%,120.77,Tighter real conditions
2026-08-20 00:00:00,+6.0,+4.0,-2.0,$89.75,16.01,2.75%,118.25,Supply / stagflation
2026-09-01 00:00:00,+6.0,+6.0,+0.0,$91.48,16.34,2.65%,118.66,Demand weakness


## 8. Export results

The final cell creates files that can be downloaded from Colab’s Files panel.

In [9]:
from pathlib import Path

data.to_csv("inflation_signal_monitor_full_data.csv")
summary.to_csv("inflation_signal_monitor_latest_readings.csv")
corr_table.to_csv("inflation_signal_monitor_correlations.csv")
episodes.to_csv("inflation_signal_monitor_largest_episodes.csv")
fig_decomp.write_html("inflation_signal_monitor_decomposition.html")
fig_cross.write_html("inflation_signal_monitor_cross_market.html")

print("Created:")
for path in sorted(Path.cwd().glob("inflation_signal_monitor_*")):
    print(" -", path.name)

Created:
 - inflation_signal_monitor_correlations.csv
 - inflation_signal_monitor_cross_market.html
 - inflation_signal_monitor_decomposition.html
 - inflation_signal_monitor_full_data.csv
 - inflation_signal_monitor_largest_episodes.csv
 - inflation_signal_monitor_latest_readings.csv


## Interpretation cautions

1. `T5YIE` is inflation compensation, not pure expected inflation.
2. The daily data cannot isolate intraday announcement responses.
3. Correlations vary with the selected sample and can reflect common shocks.
4. WTI, VIX, credit-spread, and dollar observations have different publication calendars; limited forward-filling is used only for alignment.
5. Large-movement dates identify episodes for investigation; they do not identify the news responsible for those movements.